In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} Pages from PDF")
print("--- First Page Preview (500) Character ---")
print(pages[0].page_content[:500])

Loaded 9 Pages from PDF
--- First Page Preview (500) Character ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
print(pages[2].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(pages)

print("Length of Chunks: ", len(chunks))

Length of Chunks:  37


In [8]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [9]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready {vector_store._collection.count()} vector stores.")

F:\machine-learning\Agentic-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1186.99it/s]


Vector store ready 37 vector stores.


In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "What is VolTe and how does it improve call quality?"
retrieved = retriever.invoke(test_query)
for i, doc in enumerate(retrieved):
    print(i)
    print(doc.page_content)
    print(len(doc.page_content))
    print()

0
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
595

1
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent
the device may not support VoLTE or

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n----\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT="""\
You are a helpful Telecom Assistant
Answer the question using ONLY provided context
if the context does not contain enough information, say so clearly

Context: 
{context}
"""

prompt = ChatPromptTemplate(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain Assembled")

RAG Chain Assembled


In [22]:
question = "How does the internation roaming work ?"

print(f"Q: {question}\n")
print("A: ", chain.invoke(question))

Q: How does the internation roaming work ?

A:  International roaming allows a customer's device to connect to a partner network in a foreign country when outside the home network's coverage. Here's how it works based on the provided context:

1. **Connection to Partner Network**: When traveling abroad, the device automatically connects to a pre-arranged partner network in the visited country.  
2. **Authentication & Authorization**:  
   - The visited network authenticates the user via inter-operator signaling protocols (SS7 or Diameter).  
   - The home network validates the subscription and authorizes services (voice, data, SMS).  
3. **Network Steering**:  
   - The SIM uses priority lists to steer the device to preferred partner networks. If the preferred network is unavailable, it tries the next on the list.  
   - Manual network selection in device settings can override this steering.  
4. **Activation Requirements**:  
   - Roaming must be enabled on the account before travel (